# RabTech Academy — Task 05
## Deep Learning / NLP Text Classifier with TensorFlow
### Movie Review Sentiment Classification

**Pipeline:** text → TF-IDF → multi-layer neural network → POSITIVE/NEGATIVE + confidence.

Includes Batch Normalization, Dropout, Early Stopping, convergence curves, test evaluation and unseen-text inference.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import joblib

SEED=42
np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)


## 1. Load and decode the Keras IMDB dataset

In [ ]:
VOCAB_LIMIT=20000
(x_seq,y_full),(x_test_seq,y_test)=keras.datasets.imdb.load_data(num_words=VOCAB_LIMIT)
word_index=keras.datasets.imdb.get_word_index()
reverse={i+3:w for w,i in word_index.items()}
reverse.update({0:"<PAD>",1:"<START>",2:"<UNK>",3:"<UNUSED>"})

def decode(seq):
    return " ".join(reverse.get(i,"<UNK>") for i in seq if i>3)

texts=[decode(s) for s in x_seq]
test_texts=[decode(s) for s in x_test_seq]
print(len(texts), "training reviews;", len(test_texts), "test reviews")
print(texts[0][:600])


## 2. Training/validation split

In [ ]:
train_texts,val_texts,y_train,y_val=train_test_split(
    texts,y_full,test_size=.20,random_state=SEED,stratify=y_full)
print(len(train_texts),len(val_texts),len(test_texts))


## 3. TF-IDF — fitted on training text only

In [ ]:
MAX_FEATURES=10000
vectorizer=TfidfVectorizer(max_features=MAX_FEATURES,ngram_range=(1,2),
                           min_df=2,max_df=.95,sublinear_tf=True,dtype=np.float32)
X_train=vectorizer.fit_transform(train_texts)
X_val=vectorizer.transform(val_texts)
X_test=vectorizer.transform(test_texts)
print(X_train.shape,X_val.shape,X_test.shape)


## 4. Multi-layer neural network with Batch Normalization and Dropout

In [ ]:
model=keras.Sequential([
    layers.Input(shape=(MAX_FEATURES,)),
    layers.Dense(256,activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(.50),
    layers.Dense(128,activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(.40),
    layers.Dense(64,activation="relu"),
    layers.Dropout(.30),
    layers.Dense(1,activation="sigmoid")
])
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy",metrics=["accuracy"])
model.summary()


## 5. Train with Early Stopping

In [ ]:
early=keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,
                                           restore_best_weights=True,verbose=1)
history=model.fit(X_train,y_train,validation_data=(X_val,y_val),
                  epochs=20,batch_size=128,callbacks=[early],verbose=1)


## 6. Loss and accuracy convergence curves

In [ ]:
pd.DataFrame(history.history).display() if hasattr(pd.DataFrame(history.history),'display') else display(pd.DataFrame(history.history))
plt.figure(figsize=(8,5))
plt.plot(history.history["loss"],label="Training Loss")
plt.plot(history.history["val_loss"],label="Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training vs Validation Loss")
plt.legend(); plt.grid(alpha=.25); plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["accuracy"],label="Training Accuracy")
plt.plot(history.history["val_accuracy"],label="Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Training vs Validation Accuracy")
plt.legend(); plt.grid(alpha=.25); plt.show()


## 7. Held-out test evaluation

In [ ]:
loss,acc=model.evaluate(X_test,y_test,verbose=0)
prob=model.predict(X_test,batch_size=256,verbose=0).ravel()
pred=(prob>=.5).astype(int)
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {acc:.4f}")
print(classification_report(y_test,pred,target_names=["NEGATIVE","POSITIVE"],digits=4))
ConfusionMatrixDisplay(confusion_matrix(y_test,pred),
                       display_labels=["NEGATIVE","POSITIVE"]).plot()
plt.title("Sentiment Classifier — Confusion Matrix"); plt.show()


## 8. Unseen text inference with confidence

In [ ]:
def predict_sentiment(text):
    p=float(model.predict(vectorizer.transform([text]),verbose=0)[0][0])
    label="POSITIVE" if p>=.5 else "NEGATIVE"
    confidence=p if p>=.5 else 1-p
    return {"text":text,"prediction":label,"confidence_percent":round(confidence*100,2)}

samples=[
"This movie was absolutely amazing. The acting was brilliant and I loved every minute.",
"I hated this film. The story was boring and a complete waste of time.",
"The movie had some good moments, but overall it was disappointing.",
"Fantastic performances and a beautiful story. I would definitely watch it again."
]
display(pd.DataFrame([predict_sentiment(x) for x in samples]))


In [ ]:
my_review="The story was engaging and the performances were excellent."
r=predict_sentiment(my_review)
print("Text:",r["text"])
print("Prediction:",r["prediction"])
print(f'Confidence: {r["confidence_percent"]}%')


## 9. Save trained artifacts

In [ ]:
model.save("sentiment_model.keras")
joblib.dump(vectorizer,"tfidf_vectorizer.joblib")
print("Saved sentiment_model.keras")
print("Saved tfidf_vectorizer.joblib")


## Conclusion
The project implements the required TF-IDF NLP pipeline and a multi-layer TensorFlow network with Batch Normalization, Dropout and Early Stopping. The notebook reports training/validation convergence and demonstrates predictions with confidence on unseen text.

**Leakage control:** TF-IDF is fitted only on training text; the official test set is held out until final evaluation.

**Limitation:** sentiment is context-dependent and confidence scores are not certainty.
